# 🧠 Preprocessing Pipeline — Debug & Analysis Notebook

This notebook is your debug workbench for the hybrid preprocessing pipeline.
It lets you run both the rule-based and LLM classifiers on individual PDFs
and inspect their outputs side-by-side before running the full pipeline.

---

## 📐 Pipeline Architecture

```
RAW PDFs (data/raw_docs/)
         │
         ▼
  ┌──────────────────────────────┐
  │  STAGE 1: Text Extraction   │   pdfplumber → PyMuPDF fallback
  │  Adaptive: 3 → 5 → 10 pages│
  └──────────────┬───────────────┘
                 │
                 ▼
  ┌──────────────────────────────┐
  │  STAGE 2: Rule-Based Detect  │   detect_bank() + detect_card()
  │  Fast, deterministic, free   │   + detect_doc_type() + detect_master_doc()
  └──────────────┬───────────────┘
                 │
         ┌───────┴──────────┐
         │                  │
  conf >= 0.70         conf < 0.70
         │                  │
         │                  ▼
         │   ┌──────────────────────────────┐
         │   │  STAGE 3: LLM Fallback       │  LLaMA 3 via Ollama (local)
         │   │  Semantic, flexible          │  classify_with_llm(text)
         │   └──────────────┬───────────────┘
         │                  │
         │          ┌───────┴──────────┐
         │    LLM conf > rule          LLM conf ≤ rule
         │    + 0.10 margin            + 0.10 margin
         │          │                  │
         │    [OVERRIDE]         [KEEP RULES]
         │          │                  │
         └──────────┴──────────────────┘
                          │
                          ▼
  ┌──────────────────────────────┐
  │  STAGE 4: Normalise + Save  │
  │  • Rename file              │
  │  • Copy to processed_docs/  │
  │  • Write metadata JSON      │
  └──────────────────────────────┘
```

---

## 🔀 Hybrid Logic — Rule vs LLM

| Condition | Action |
|-----------|--------|
| Rule confidence ≥ 0.70 | Skip LLM, use rule result |
| Rule confidence < 0.70 AND Ollama available | Call LLM |
| LLM confidence > rule + 0.10 | Override with LLM result |
| LLM confidence ≤ rule + 0.10 | Keep rule result (LLM not better enough) |
| LLM call fails / timeout | Keep rule result, log error |

**Why the 0.10 margin?** Rules are deterministic and domain-tuned. We need the LLM to clearly outperform rules before trusting it. A 10% confidence margin prevents the LLM from winning on noise.

---

## 📦 Expected Outputs

For each processed PDF:
```
data/processed_docs/HDFC_Millennia/
    HDFC_Millennia_MITC_2026.pdf        ← renamed PDF
    HDFC_Millennia_MITC_2026.json       ← metadata for RAG
```

Metadata JSON structure:
```json
{
  "bank": "HDFC",
  "card": "Millennia",
  "doc_type": "MITC",
  "is_master": false,
  "confidence": 0.91,
  "classification_source": "rule_based",
  "source_file": "raw_hdfc_millennia_mitc.pdf",
  "output_file": "HDFC_Millennia_MITC_2026.pdf",
  "llm_reason": null,
  "processing_timestamp": "2026-01-15T10:30:00"
}
```

---

## ⚠️ Known Failure Cases

| Failure Case | Symptom | Fix |
|---|---|---|
| Scanned PDF (image-only) | No text extracted, all confidence = 0 | Add OCR (e.g. pytesseract) |
| New bank not in BANK_ALIASES | bank=UNKNOWN | Add bank to BANK_ALIASES in preprocess.py |
| New card not in CARDS list | card=UNKNOWN | Add card to CARDS list in preprocess.py |
| Ollama not running | LLM skipped, falls back to rules | Run: ollama serve |
| LLM returns UNKNOWN for bank | LLM bank ignored, rule bank kept | Expected — rules are better at bank detection |
| Collective doc detected as card-specific | Wrong card name | Check MASTER_DOC_SIGNALS in preprocess.py |

## ⚙️ Setup — Imports & Path Configuration

In [ ]:
import sys
import json
import logging
from pathlib import Path

# ─── Add data_pipeline to path so we can import preprocess.py and llm_classifier.py ───
# Adjust this if your notebook is in a different folder.
NOTEBOOK_DIR  = Path().resolve()
PROJECT_ROOT  = NOTEBOOK_DIR.parent        # assumes notebook is in /notebooks/
PIPELINE_DIR  = PROJECT_ROOT / 'data_pipeline'
RAW_DOCS_DIR  = PROJECT_ROOT / 'data' / 'raw_docs'

sys.path.insert(0, str(PIPELINE_DIR))

print(f'Project root : {PROJECT_ROOT}')
print(f'Pipeline dir : {PIPELINE_DIR}')
print(f'Raw docs dir : {RAW_DOCS_DIR}')
print(f'Exists?      : {RAW_DOCS_DIR.exists()}')

# Configure logging so we see INFO messages in notebook output
logging.basicConfig(
    level   = logging.INFO,
    format  = '[%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
print('Setup complete.')

## 📥 Step 1 — Load Sample PDFs

In [ ]:
from preprocess import extract_text, PAGE_TIERS

# ─── List available PDFs ──────────────────────────────────────────────────────
pdf_files = sorted(RAW_DOCS_DIR.glob('*.pdf'))
print(f'Found {len(pdf_files)} PDF(s) in raw_docs/:')
for i, p in enumerate(pdf_files):
    print(f'  [{i}] {p.name}')

if not pdf_files:
    print('\n⚠️  No PDFs found. Place PDF files in data/raw_docs/ and re-run this cell.')

In [ ]:
# ─── Select a PDF to debug ────────────────────────────────────────────────────
# Change the index to test different files.
# Example: SAMPLE_PDF_INDEX = 0 → tests the first PDF in the list.

SAMPLE_PDF_INDEX = 0   # ← Change this to test different PDFs

if not pdf_files:
    print('No PDFs available. Skipping.')
else:
    sample_pdf = pdf_files[SAMPLE_PDF_INDEX]
    print(f'Selected: {sample_pdf.name}')

    # Extract text using the same adaptive strategy as the pipeline
    # PAGE_TIERS = [3, 5, 10] — try 3 pages first
    sample_text = extract_text(sample_pdf, max_pages=PAGE_TIERS[0])

    if sample_text.strip():
        print(f'\nExtracted {len(sample_text)} characters ({PAGE_TIERS[0]} pages).')
        print('\n── First 800 characters ──────────────────────────────────────────')
        print(sample_text[:800])
        print('──────────────────────────────────────────────────────────────────')
    else:
        print('⚠️  No text extracted. This may be a scanned (image-only) PDF.')

## 🔍 Step 2 — Rule-Based Classification

In [ ]:
from preprocess import (
    detect_bank,
    detect_card,
    detect_doc_type,
    detect_master_doc,
    detect_year,
    compute_confidence,
    CONFIDENCE_THRESHOLD,
)

if 'sample_text' not in dir() or not sample_text.strip():
    print('No text available. Run Step 1 first.')
else:
    print('[STEP 2] Running rule-based classification...')
    print()

    # Run each detection function
    rule_bank_result     = detect_bank(sample_text, sample_pdf.name)
    rule_card_result     = detect_card(sample_text, sample_pdf.name)
    rule_doc_type_result = detect_doc_type(sample_text)
    rule_master_result   = detect_master_doc(sample_text)
    rule_year            = detect_year(sample_text)

    rule_confidence = compute_confidence(
        doc_type_result = rule_doc_type_result,
        bank_result     = rule_bank_result,
        card_result     = rule_card_result,
        master_result   = rule_master_result,
    )

    # ─── Print structured output ──────────────────────────────────────────────
    print('═' * 55)
    print('RULE-BASED CLASSIFICATION RESULT')
    print('═' * 55)
    print(f'  Bank      : {rule_bank_result["value"]:20s}  (conf={rule_bank_result["confidence"]:.2f})')
    print(f'  Card      : {str(rule_card_result["value"]):20s}  (conf={rule_card_result["confidence"]:.2f})')
    print(f'  Doc Type  : {rule_doc_type_result["value"]:20s}  (conf={rule_doc_type_result["confidence"]:.2f})')
    print(f'  Is Master : {str(rule_master_result["is_master"]):20s}  (conf={rule_master_result["confidence"]:.2f})')
    print(f'  Year      : {rule_year}')
    print(f'  Overall   : {rule_confidence:.3f}')
    print()

    threshold_met = rule_confidence >= CONFIDENCE_THRESHOLD
    status_icon   = '✅' if threshold_met else '⚠️ '
    print(f'  {status_icon} Threshold ({CONFIDENCE_THRESHOLD}) : {"MET — LLM not needed" if threshold_met else "NOT MET — LLM fallback will be triggered"}')
    print('═' * 55)

    print('\nDetailed reasons:')
    print('  Bank reasons:')
    for r in rule_bank_result['reasons']:     print(f'    • {r}')
    print('  Card reasons:')
    for r in rule_card_result['reasons']:     print(f'    • {r}')
    print('  Doc type reasons:')
    for r in rule_doc_type_result['reasons']: print(f'    • {r}')

## 🤖 Step 3 — LLM Classification (LLaMA 3 via Ollama)

In [ ]:
# ─── Ollama availability check ────────────────────────────────────────────────
# Before calling the LLM, verify Ollama is running.
# If this fails, follow the setup instructions below.

from llm_classifier import check_ollama_available, classify_with_llm

ollama_ok = check_ollama_available()
if ollama_ok:
    print('✅ Ollama is running. Proceeding with LLM classification.')
else:
    print('❌ Ollama not available.')
    print()
    print('SETUP INSTRUCTIONS:')
    print('  1. Install Ollama:')
    print('     macOS/Linux: curl -fsSL https://ollama.ai/install.sh | sh')
    print('     Windows: Download from https://ollama.ai/download')
    print()
    print('  2. Pull LLaMA 3 model (~4.7 GB, one-time download):')
    print('     ollama pull llama3')
    print()
    print('  3. Start Ollama server:')
    print('     ollama serve')
    print()
    print('  Then re-run this cell.')

In [ ]:
# ─── Run LLM classification ───────────────────────────────────────────────────
# This cell runs regardless of rule confidence — useful for comparing
# LLM output against rules even on high-confidence documents.

if not ollama_ok:
    print('Skipping LLM call — Ollama not available. See setup instructions above.')
    llm_result = None
elif 'sample_text' not in dir() or not sample_text.strip():
    print('No text available. Run Step 1 first.')
    llm_result = None
else:
    print('[STEP 3] Calling LLaMA 3 via Ollama...')
    print('(This may take 10–30 seconds on CPU, 2–5s on GPU)')
    print()

    llm_result = classify_with_llm(sample_text)

    print()
    print('═' * 55)
    print('LLM CLASSIFICATION RESULT (LLaMA 3)')
    print('═' * 55)

    if llm_result['llm_success']:
        print(f'  Bank      : {llm_result["bank"]}')
        print(f'  Card      : {llm_result["card_name"]}')
        print(f'  Doc Type  : {llm_result["doc_type"]}')
        print(f'  Is Master : {llm_result["is_master"]}')
        print(f'  Confidence: {llm_result["confidence"]:.3f}')
        print(f'  Reason    : {llm_result["reason"]}')
        print()
        print('[LLM OUTPUT]', json.dumps(llm_result, indent=2))
    else:
        print('  ❌ LLM classification failed.')
        print(f'  Reason: {llm_result["reason"]}')

    print('═' * 55)

## ⚖️ Step 4 — Compare Rule vs LLM Output

In [ ]:
# ─── Side-by-side comparison ──────────────────────────────────────────────────
# This cell shows you exactly what the hybrid decision logic would do.

from preprocess_with_llm import apply_llm_override, LLM_OVERRIDE_MARGIN

if 'rule_confidence' not in dir():
    print('Run Step 2 first.')
elif llm_result is None:
    print('LLM result not available. Showing rule result only.')
else:
    print('═' * 65)
    print('COMPARISON — RULE vs LLM')
    print('═' * 65)
    print(f'{'Field':<15} {'RULE':^25} {'LLM':^25}')
    print('─' * 65)
    print(f'{'Bank':<15} {str(rule_bank_result["value"]):^25} {str(llm_result.get("bank","N/A")):^25}')
    print(f'{'Card':<15} {str(rule_card_result["value"]):^25} {str(llm_result.get("card_name","N/A")):^25}')
    print(f'{'Doc Type':<15} {str(rule_doc_type_result["value"]):^25} {str(llm_result.get("doc_type","N/A")):^25}')
    print(f'{'Is Master':<15} {str(rule_master_result["is_master"]):^25} {str(llm_result.get("is_master","N/A")):^25}')
    print(f'{'Confidence':<15} {rule_confidence:^25.3f} {llm_result.get("confidence", 0):^25.3f}')
    print('═' * 65)

    # ─── Simulate the hybrid decision ─────────────────────────────────────────
    final_bank, final_card, final_doc_type, final_is_master, final_conf, src = \
        apply_llm_override(
            rule_bank       = rule_bank_result['value'],
            rule_card       = rule_card_result['value'],
            rule_doc_type   = rule_doc_type_result['value'],
            rule_is_master  = rule_master_result['is_master'],
            rule_confidence = rule_confidence,
            llm_result      = llm_result,
        )

    print()
    icon = '🤖' if src == 'llm' else '📐'
    print(f'{icon} HYBRID DECISION: Using "{src.upper()}" result')
    print(f'   Final Bank     : {final_bank}')
    print(f'   Final Card     : {final_card}')
    print(f'   Final Doc Type : {final_doc_type}')
    print(f'   Final Master   : {final_is_master}')
    print(f'   Final Conf     : {final_conf:.3f}')
    print()

    margin_needed = rule_confidence + LLM_OVERRIDE_MARGIN
    llm_conf      = llm_result.get('confidence', 0)
    print(f'   Override threshold : rule_conf + margin = {rule_confidence:.2f} + {LLM_OVERRIDE_MARGIN} = {margin_needed:.2f}')
    print(f'   LLM confidence     : {llm_conf:.2f}')
    print(f'   LLM wins?          : {llm_conf} > {margin_needed:.2f} → {llm_conf > margin_needed}')

## 🧾 Step 5 — Generate Metadata JSON Preview

In [ ]:
# ─── Preview the metadata JSON that would be saved alongside the PDF ──────────
# This is what the RAG system will read to attach structured metadata
# to each text chunk during vector DB indexing.

from preprocess_with_llm import generate_metadata
from preprocess import generate_filename

if 'final_bank' not in dir():
    # If comparison step wasn't run, use rule results
    if 'rule_confidence' not in dir():
        print('Run Steps 2-4 first.')
    else:
        final_bank      = rule_bank_result['value']
        final_card      = rule_card_result['value']
        final_doc_type  = rule_doc_type_result['value']
        final_is_master = rule_master_result['is_master']
        final_conf      = rule_confidence
        src             = 'rule_based'

if 'final_bank' in dir():
    output_filename = generate_filename(final_bank, final_card, final_doc_type, rule_year)

    metadata = generate_metadata(
        bank                  = final_bank,
        card                  = final_card,
        doc_type              = final_doc_type,
        is_master             = final_is_master,
        confidence            = final_conf,
        classification_source = src,
        source_file           = sample_pdf.name if 'sample_pdf' in dir() else 'unknown.pdf',
        llm_reason            = llm_result.get('reason') if llm_result and src == 'llm' else None,
        dest_path             = Path(output_filename),
    )

    print('METADATA JSON (will be saved alongside processed PDF):')
    print('─' * 55)
    print(json.dumps(metadata, indent=2))
    print('─' * 55)
    print(f'\nOutput filename : {output_filename}')
    print(f'JSON filename   : {output_filename.replace(".pdf", ".json")}')

## 🔁 Step 6 — Batch Test Multiple PDFs

In [ ]:
# ─── Test first N PDFs in raw_docs/ and print a summary table ─────────────────
# Useful for catching classification errors before running the full pipeline.
# Does NOT move or copy files — read-only analysis.

from preprocess import run_detection_with_fallback

TEST_N = 5   # Number of PDFs to test. Change as needed.

print(f'Testing first {TEST_N} PDF(s)...')
print('─' * 85)
print(f'{'File':<35} {'Bank':^10} {'Card':^18} {'Type':^6} {'Conf':^6} {'Source'}')
print('─' * 85)

test_files = pdf_files[:TEST_N]
if not test_files:
    print('No PDFs available.')
else:
    for pdf_path in test_files:
        try:
            text, doc_type_res, bank_res, card_res, master_res, year, rule_conf = \
                run_detection_with_fallback(pdf_path, debug=False)

            # LLM fallback (only if ollama is available AND confidence is low)
            src       = 'rule'
            llm_res   = None
            final_conf= rule_conf
            if ollama_ok and rule_conf < CONFIDENCE_THRESHOLD:
                llm_res = classify_with_llm(text)
                if llm_res and llm_res.get('llm_success') and \
                   llm_res['confidence'] > rule_conf + LLM_OVERRIDE_MARGIN:
                    src        = 'LLM'
                    final_conf = llm_res['confidence']

            bank_val = bank_res['value'] or 'UNKNOWN'
            card_val = card_res['value'] or 'UNKNOWN'
            dt_val   = doc_type_res['value'] or 'UNK'

            flag = '' if final_conf >= CONFIDENCE_THRESHOLD else ' ⚠️'
            print(f'{pdf_path.name[:34]:<35} {bank_val:^10} {str(card_val)[:17]:^18} {dt_val:^6} {final_conf:^6.2f} {src}{flag}')

        except Exception as e:
            print(f'{pdf_path.name[:34]:<35} ERROR: {e}')

    print('─' * 85)
    print('⚠️  = confidence below threshold, file would go to needs_review/')

## 🚀 Step 7 — Run Full Hybrid Pipeline

In [ ]:
# ─── Run the full hybrid pipeline ─────────────────────────────────────────────
# This processes ALL PDFs in raw_docs/ and produces all outputs.
# 
# Set DRY_RUN = True to simulate without moving any files.
# Set USE_LLM = False to run rule-based only (faster, no Ollama needed).

DRY_RUN = True    # ← Set to False for a real run
USE_LLM = True    # ← Set to False to skip LLM fallback

from preprocess_with_llm import process_all_hybrid

print(f'Running pipeline: dry_run={DRY_RUN}, use_llm={USE_LLM}')
print()

process_all_hybrid(
    dry_run = DRY_RUN,
    debug   = False,
    use_llm = USE_LLM,
)

## 📊 Step 8 — Review Results

In [ ]:
# ─── Load and display summary CSV ────────────────────────────────────────────
try:
    import pandas as pd
    summary_csv = PROJECT_ROOT / 'data' / 'logs' / 'summary.csv'
    if summary_csv.exists():
        df = pd.read_csv(summary_csv)
        print(f'Loaded {len(df)} entries from summary.csv')
        print()
        print('STATUS BREAKDOWN:')
        print(df['Status'].value_counts().to_string())
        print()
        print('SAMPLE ROWS:')
        display(df.head(10))
    else:
        print(f'summary.csv not found at {summary_csv}')
        print('Run Step 7 with DRY_RUN=False first.')
except ImportError:
    print('pandas not installed. Run: pip install pandas')

In [ ]:
# ─── Check metadata JSON files in processed_docs/ ────────────────────────────
processed_dir = PROJECT_ROOT / 'data' / 'processed_docs'
json_files    = list(processed_dir.rglob('*.json'))

print(f'Found {len(json_files)} metadata JSON file(s) in processed_docs/')

# Show the first few
for jf in json_files[:3]:
    print(f'\n── {jf.name} ──────────────────────────────────────────────')
    with open(jf) as f:
        print(json.dumps(json.load(f), indent=2))

In [ ]:
# ─── Show files that need review ──────────────────────────────────────────────
review_dir   = PROJECT_ROOT / 'data' / 'needs_review'
review_files = list(review_dir.glob('*.pdf'))

print(f'{len(review_files)} file(s) in needs_review/:')
for f in review_files:
    print(f'  • {f.name}')

if review_files:
    print()
    print('These files had confidence < 0.70 after both rule and LLM classification.')
    print('Actions:')
    print('  1. Run with --debug to see extracted text')
    print('  2. Add missing bank/card names to preprocess.py config')
    print('  3. Check if PDF is scanned (needs OCR)')